In [ ]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Loading the datasets
data_2020 = pd.read_csv('/content/countypres_2000-2020.csv')

# Displaying the first 5 rows of the dataset
display(data_2020.head())

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
0,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,AL GORE,DEMOCRAT,4942,17208,20220315,TOTAL
1,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,GEORGE W. BUSH,REPUBLICAN,11993,17208,20220315,TOTAL
2,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,RALPH NADER,GREEN,160,17208,20220315,TOTAL
3,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,OTHER,OTHER,113,17208,20220315,TOTAL
4,2000,ALABAMA,AL,BALDWIN,1003.0,US PRESIDENT,AL GORE,DEMOCRAT,13997,56480,20220315,TOTAL


In [ ]:
# Printing the column names
print(data_2020.columns)

Index(['year', 'state', 'state_po', 'county_name', 'county_fips', 'office',
       'candidate', 'party', 'candidatevotes', 'totalvotes', 'version',
       'mode'],
      dtype='object')


In [ ]:
# Checking for missing values
data_2020.dropna(inplace=True)

# Displaying the first 5 rows of the dataset
display(data_2020.head())

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
0,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,AL GORE,DEMOCRAT,4942,17208,20220315,TOTAL
1,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,GEORGE W. BUSH,REPUBLICAN,11993,17208,20220315,TOTAL
2,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,RALPH NADER,GREEN,160,17208,20220315,TOTAL
3,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,OTHER,OTHER,113,17208,20220315,TOTAL
4,2000,ALABAMA,AL,BALDWIN,1003.0,US PRESIDENT,AL GORE,DEMOCRAT,13997,56480,20220315,TOTAL


In [ ]:
# Extracting only the needed columns
# Filtering data for Pennsylvania, year 2020, and relevant parties
win_data_2020 = data_2020[
    (data_2020['year'] == 2020) &
    (data_2020['state_po'] == 'PA') &
    (data_2020['party'].isin(['DEMOCRAT', 'REPUBLICAN']))
]

# Displaying the dataset
display(win_data_2020)

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
63762,2020,PENNSYLVANIA,PA,ADAMS,42001.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,18207,56540,20220315,TOTAL
63764,2020,PENNSYLVANIA,PA,ADAMS,42001.0,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,37523,56540,20220315,TOTAL
63765,2020,PENNSYLVANIA,PA,ALLEGHENY,42003.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,429065,719733,20220315,TOTAL
63767,2020,PENNSYLVANIA,PA,ALLEGHENY,42003.0,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,282324,719733,20220315,TOTAL
63768,2020,PENNSYLVANIA,PA,ARMSTRONG,42005.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,8457,36370,20220315,TOTAL
...,...,...,...,...,...,...,...,...,...,...,...,...
63956,2020,PENNSYLVANIA,PA,WESTMORELAND,42129.0,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,130218,204697,20220315,TOTAL
63957,2020,PENNSYLVANIA,PA,WYOMING,42131.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,4704,14858,20220315,TOTAL
63959,2020,PENNSYLVANIA,PA,WYOMING,42131.0,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,9936,14858,20220315,TOTAL
63960,2020,PENNSYLVANIA,PA,YORK,42133.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,88114,238471,20220315,TOTAL


In [ ]:
# Ensureing that the `candidatevotes` column is numeric
win_data_2020['candidatevotes'] = pd.to_numeric(win_data_2020['candidatevotes'], errors='coerce')

# Createing a pivot table to summarize votes by county and party
win_data_2020 = win_data_2020.pivot_table(index='county_name', columns='party', values='candidatevotes', aggfunc='sum').fillna(0)

# Renameing the columns for clarity
win_data_2020 = win_data_2020.rename(columns={'DEMOCRAT': 'dem_votes', 'REPUBLICAN': 'rep_votes'})

# Calculating the absolute win margin and win margin percentage
win_data_2020['win_margin_abs'] = abs(win_data_2020['dem_votes'] - win_data_2020['rep_votes'])
win_data_2020['win_margin_pct'] = (win_data_2020['win_margin_abs'] / (win_data_2020['dem_votes'] + win_data_2020['rep_votes'])) * 100

# Determining the winning party
win_data_2020['winning_party'] = win_data_2020.apply(lambda x: 'Democrat' if x['dem_votes'] > x['rep_votes'] else 'Republican', axis=1)

# Resetting index to turn county_name into a column
win_data_2020 = win_data_2020.reset_index()

# Displaying the dataset
display(win_data_2020.head())

<ipython-input-26-2d845536a0e7>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  win_data_2020['candidatevotes'] = pd.to_numeric(win_data_2020['candidatevotes'], errors='coerce')


party,county_name,dem_votes,rep_votes,win_margin_abs,win_margin_pct,winning_party
0,ADAMS,18207,37523,19316,34.659968,Republican
1,ALLEGHENY,429065,282324,146741,20.627392,Democrat
2,ARMSTRONG,8457,27489,19032,52.946086,Republican
3,BEAVER,38122,54759,16637,17.912167,Republican
4,BEDFORD,4367,23025,18658,68.114778,Republican


In [ ]:
# Renaming the columns
win_data_2020 = win_data_2020.rename(columns={
    'county_name': 'countyname',
    'dem_votes': 'party_dem',
    'rep_votes': 'party_rep'
})

# Displaying the updated DataFrame to confirm the changes
display(win_data_2020.head())

# Saving the updated dataset to a CSV file
win_data_2020.to_csv('elections_results_2020.csv', index=False)


party,countyname,party_dem,party_rep,win_margin_abs,win_margin_pct,winning_party
0,ADAMS,18207,37523,19316,34.659968,Republican
1,ALLEGHENY,429065,282324,146741,20.627392,Democrat
2,ARMSTRONG,8457,27489,19032,52.946086,Republican
3,BEAVER,38122,54759,16637,17.912167,Republican
4,BEDFORD,4367,23025,18658,68.114778,Republican
